# EP.3 位置编码：从等变性到 RoPE 与长度缩放

配套文章：[位置编码技术的演进](https://momoyeyu.github.io/posts/llm-position-encoding/)。上一章：[02_normalization.ipynb](02_normalization.ipynb)；下一章：[04_ppo.ipynb](04_ppo.ipynb)。本章复用 [EP.0](00_transformer.ipynb) 首次展开的 `MHA`、`PositionEncoding`、`attention`、`causal_mask` 与 `set_seed` / `show`，逐步加入绝对、相对、线性 bias 与旋转位置。重点区分“公式可计算”“结构有位置线索”和“模型能泛化”三件事。

In [1]:
from pathlib import Path
import sys

root = Path.cwd()
if not (root / "slm").is_dir():
    root = next((parent for parent in root.parents if (parent / "slm").is_dir()), None)
if root is None:
    raise RuntimeError("请从仓库根目录或 notebooks 目录运行")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from slm import set_seed, show

torch.set_num_threads(1)
set_seed(0)
show("Python", sys.version.split()[0])
show("PyTorch", torch.__version__)
show("device", "cpu")
from types import SimpleNamespace
from slm import MHA, PositionEncoding, attention, causal_mask

Python                       3.12.12
PyTorch                      2.14.1
device                       cpu


## 1. 没有位置时的 permutation equivariance

重排输入 token 后，输出按相同方式重排，这叫等变，不是输出完全不变的 invariance。

In [2]:
import torch


set_seed(0)
d_model, num_heads, L = 16, 4, 8
mha = MHA(d_model, num_heads)                              # MHA 为 EP.0 中实现的类
x = torch.randn(1, L, d_model)                             # [1, L, d_model]
out = mha(x, x, x)                                         # [1, L, d_model]
x_flipped = x.flip(1)                                      # [1, L, d_model]
out_shuffled = mha(x_flipped, x_flipped, x_flipped)
show("permutation equivariant", torch.allclose(out_shuffled, out.flip(1), atol=1e-5))

permutation equivariant      True


实验只验证无 mask MHA 的结构性质；加入位置向量后，同一内容被分配到不同位置向量，因此不再满足同样等式。

## 2. 可学习绝对位置与正弦位置

LearnedPE 查表，超过 `max_len` 会越界；正弦位置可在缓存外继续按公式计算。

In [3]:
class LearnedPE(nn.Module):
    def __init__(self, max_len: int, d_model: int) -> None:
        super().__init__()
        self.pos_emb = nn.Embedding(max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, L, d_model]，已含 token embedding
        return x + self.pos_emb(torch.arange(x.size(1), device=x.device))  # [B,L,d_model] + [L,d_model]

In [4]:
learned = LearnedPE(8, 16)
zeros = torch.zeros(1, 8, 16)
show('learned table matches', torch.allclose(learned(zeros)[0], learned.pos_emb(torch.arange(8))))
sinusoidal = PositionEncoding(16, 8)
show('sinusoidal shape', tuple(sinusoidal(zeros).shape))

learned table matches        True
sinusoidal shape             (1, 8, 16)


LearnedPE 的硬长度限制清晰可见；sin/cos 在更长位置仍能算出数值，但这不自动带来长度外推质量。

## 3. 相对距离：Shaw 向量与 T5 标量 bias

先把有符号距离 `j-i` 截断并平移成 embedding 下标。

In [5]:
def rel_index(L: int, max_dist: int, device: torch.device | None = None) -> torch.Tensor:
    i = torch.arange(L, device=device)[:, None]   # [L, 1]
    j = torch.arange(L, device=device)[None, :]   # [1, L]
    return (j - i).clamp(-max_dist, max_dist) + max_dist  # [L, L]

In [6]:
class ShawRelBias(nn.Module):
    """Shaw et al. 2018：每个相对距离一个可学习向量，加到 key 上。"""

    def __init__(self, max_dist: int, d_head: int) -> None:
        super().__init__()
        self.max_dist = max_dist
        self.rel_k = nn.Embedding(2 * max_dist + 1, d_head)

    def forward(self, q: torch.Tensor, k: torch.Tensor) -> torch.Tensor:
        # q, k: [H, L, d_head]
        rel = rel_index(q.size(1), self.max_dist, q.device)  # [L, L]
        return (q[:, :, None] * (k[:, None] + self.rel_k(rel))).sum(-1) / math.sqrt(q.size(-1))  # [H, L, L]

In [7]:
class T5RelBias(nn.Module):
    """T5：每个头在每个距离桶上学一个标量，直接加在 logits 上。"""

    def __init__(self, max_dist: int, num_heads: int) -> None:
        super().__init__()
        self.max_dist = max_dist
        self.rel_b = nn.Embedding(2 * max_dist + 1, num_heads)

    def forward(self, q: torch.Tensor, k: torch.Tensor) -> torch.Tensor:
        # q, k: [H, L, d_head]
        rel = rel_index(q.size(1), self.max_dist, q.device)  # [L, L]
        return q @ k.transpose(-1, -2) / math.sqrt(q.size(-1)) + self.rel_b(rel).permute(2, 0, 1)  # [H, L, L]

In [8]:
import torch


set_seed(0)
L, H, D, max_dist = 6, 2, 8, 4

rel = rel_index(L, max_dist)
show("relative distance indices", rel)

q, k = torch.randn(H, L, D), torch.randn(H, L, D)

shaw = ShawRelBias(max_dist, D)
show("Shaw logits shape", tuple(shaw(q, k).shape))

t5 = T5RelBias(max_dist, H)
bias = t5.rel_b(rel).permute(2, 0, 1)  # [H, L, L]
show("T5 logits shape", tuple(t5(q, k).shape))
# T5 的 bias 只依赖 j-i：对角线上元素相同
show("T5 bias main diagonal", bias[0].diagonal().unique())

relative distance indices    tensor([[4, 5, 6, 7, 8, 8],
        [3, 4, 5, 6, 7, 8],
        [2, 3, 4, 5, 6, 7],
        [1, 2, 3, 4, 5, 6],
        [0, 1, 2, 3, 4, 5],
        [0, 0, 1, 2, 3, 4]])
Shaw logits shape            (2, 6, 6)
T5 logits shape              (2, 6, 6)
T5 bias main diagonal        tensor([-0.6232], grad_fn=<Unique2Backward0>)


这是简化的 clipped signed-distance 教学版本，不是真实 T5 的对数分桶，也不是 Shaw 同时含 value-relative 项的完整模型。配套文章讨论了这些形式的重参数化关系。

## 4. ALiBi：把距离先验直接加到 logits

每个头使用不同斜率，bias 为 $-m_h|i-j|$。

In [9]:
def alibi_slopes(num_heads: int) -> torch.Tensor:
    return torch.pow(2.0, -torch.arange(1, num_heads + 1) * (8.0 / num_heads))  # [H]

In [10]:
def alibi_bias(L: int, num_heads: int, device: torch.device | None = None) -> torch.Tensor:
    i = torch.arange(L, device=device)[:, None]       # [L, 1]
    j = torch.arange(L, device=device)[None, :]       # [1, L]
    slopes = alibi_slopes(num_heads).to(device)       # [H]
    return -slopes[:, None, None] * (j - i).abs()     # [H, L, L]

In [11]:
import math


import torch


set_seed(0)
L, H, D = 8, 8, 16

slopes = alibi_slopes(H)
show("head slopes", slopes.tolist())

bias = alibi_bias(L, H)
show("head 0 bias", bias[0])

# causal 场景下 -m·|i-j| 与论文写法 -m·(i-j) 等价（j>i 的部分反正被 mask 掉）
i = torch.arange(L)[:, None]
j = torch.arange(L)[None, :]
visible = i >= j
causal_form = -slopes[:, None, None] * (i - j)
show("causal forms match", torch.equal(bias.masked_select(visible), causal_form.masked_select(visible)))

# 距离越远惩罚越大：近处多注意、远处少注意
q, k = torch.randn(H, L, D), torch.randn(H, L, D)
logits = q @ k.transpose(-1, -2) / math.sqrt(D) + bias
show("logits shape", tuple(logits.shape))

head slopes                  [0.5, 0.25, 0.125, 0.0625, 0.03125, 0.015625, 0.0078125, 0.00390625]
head 0 bias                  tensor([[-0.0000, -0.5000, -1.0000, -1.5000, -2.0000, -2.5000, -3.0000, -3.5000],
        [-0.5000, -0.0000, -0.5000, -1.0000, -1.5000, -2.0000, -2.5000, -3.0000],
        [-1.0000, -0.5000, -0.0000, -0.5000, -1.0000, -1.5000, -2.0000, -2.5000],
        [-1.5000, -1.0000, -0.5000, -0.0000, -0.5000, -1.0000, -1.5000, -2.0000],
        [-2.0000, -1.5000, -1.0000, -0.5000, -0.0000, -0.5000, -1.0000, -1.5000],
        [-2.5000, -2.0000, -1.5000, -1.0000, -0.5000, -0.0000, -0.5000, -1.0000],
        [-3.0000, -2.5000, -2.0000, -1.5000, -1.0000, -0.5000, -0.0000, -0.5000],
        [-3.5000, -3.0000, -2.5000, -2.0000, -1.5000, -1.0000, -0.5000, -0.0000]])
causal forms match           True
logits shape                 (8, 8, 8)


这里的几何斜率日程对应 canonical power-of-two 头数；原论文对非 2 次幂头数另有构造。距离 bias 只是先验，内容 logits 仍可能让远 token 获得更高权重。

## 5. RoPE：旋转后让点积依赖相对位置

实现采用 split-half 配对：前半维与后半维成二维旋转对，因此 D 必须为偶数。

In [12]:
def rope_matrix(pos: int, d_head: int, base: float = 10000.0) -> torch.Tensor:
    inv_freq = base ** (-torch.arange(0, d_head, 2).float() / d_head)  # [d_head/2]
    R = torch.zeros(d_head, d_head)                                    # [d_head, d_head]
    half = d_head // 2
    for i, theta in enumerate(pos * inv_freq):     # 第 i 维与第 i+d_head/2 维配对
        c, s = theta.cos(), theta.sin()
        R[i,        i] = c; R[i,        i + half] = -s
        R[i + half, i] = s; R[i + half, i + half] = c
    return R                                             # [d_head, d_head]，R(pos)

In [13]:
def precompute_cos_sin(L: int, d_head: int, base: float = 10000.0, pos_scale: float = 1.0):
    inv_freq = base ** (-torch.arange(0, d_head, 2).float() / d_head)  # [d_head/2]
    pos = torch.arange(L).float()                                    # [L]
    angles = (pos[:, None] / pos_scale) * inv_freq[None, :]            # [L, d_head/2]
    return angles.cos(), angles.sin()

In [14]:
def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """
    x: [..., d_head]
    cos/sin: [..., d_head/2]
    """
    x1, x2 = x.chunk(2, dim=-1)                          # [..., d_head/2], [..., d_head/2]
    return torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1)  # [..., d_head]

In [15]:
import torch


set_seed(0)
D = 64

# 旋转矩阵的两条支点性质：R(θ)ᵀ = R(-θ)，R(θ1)R(θ2) = R(θ1+θ2)
R3, R5, R8, Rm3 = rope_matrix(3, D), rope_matrix(5, D), rope_matrix(8, D), rope_matrix(-3, D)
show("R(3).T = R(-3)", torch.allclose(R3.T, Rm3, atol=1e-6))
show("R(3)R(5) = R(8)", torch.allclose(R3 @ R5, R8, atol=1e-6))

# 矩阵形式与逐元素形式一致
cos, sin = precompute_cos_sin(L=512, d_head=D)   # [L, d_head/2], [L, d_head/2]
x = torch.randn(D)
m = 100
show("apply_rope matches matrix", torch.allclose(apply_rope(x, cos[m], sin[m]), rope_matrix(m, D) @ x, atol=1e-5))
show("rotation preserves norm", torch.allclose(apply_rope(x, cos[m], sin[m]).norm(), x.norm(), atol=1e-5))

# 核心性质：内积只依赖 m-n，与绝对位置无关
q, k = torch.randn(D), torch.randn(D)
n = 37
lhs = apply_rope(q, cos[m], sin[m]) @ apply_rope(k, cos[n], sin[n])
rhs = (rope_matrix(m - n, D) @ q) @ k             # (R_{m-n} q) · k
show("relative dot product", torch.allclose(lhs, rhs, atol=1e-5))

# 同样的 q,k 平移到另一组绝对位置，只要 m-n 相同内积就不变
m2, n2 = 300, 237  # m2 - n2 = 63 = m - n
lhs2 = apply_rope(q, cos[m2], sin[m2]) @ apply_rope(k, cos[n2], sin[n2])
show("translation invariant dot", torch.allclose(lhs, lhs2, atol=1e-5))

R(3).T = R(-3)               True
R(3)R(5) = R(8)              True
apply_rope matches matrix    True
rotation preserves norm      True
relative dot product         True
translation invariant dot    False


正交旋转保持模长；在 q、k 内容固定时，位置 m/n 的点积可写成相对旋转。若内容随位置变化，不能把所有差异都归因于相对距离。

### QK-Norm 与 RoPE 的组合

复用 [EP.2](02_normalization.ipynb) 首次展开的 `RMSNorm`：先归一化 Q/K，再旋转；正交旋转不改变已经控制好的向量模长。

In [16]:
from slm import RMSNorm
set_seed(0)
q, k = torch.randn(2, 4, 8, 16) * 20, torch.randn(2, 4, 8, 16) * 20
cos, sin = precompute_cos_sin(8, 16)
qn, kn = RMSNorm(16)(q), RMSNorm(16)(k)
qr, kr = apply_rope(qn, cos, sin), apply_rope(kn, cos, sin)
show('Q norm preserved by RoPE', torch.allclose(qr.norm(dim=-1), qn.norm(dim=-1), atol=1e-5))
show('normalized rotated logits', (qr @ kr.transpose(-1,-2) / 4).abs().max().item())

Q norm preserved by RoPE     True
normalized rotated logits    2.657182216644287


第一项应为 True；第二项是当前随机样本的测量，不把它写成固定预期数值。learned gamma 会进一步改变尺度。

## 6. PI、NTK scaling 与滑窗 PPL

PI 把位置除以 s，因此所有频率角度都缩小；NTK base scaling 保持最高频不变，并让最低频约缩小到 1/s。公式要求 `d_head > 2`。

In [17]:
def ntk_base(base: float, d_head: int, s: float) -> float:
    return base * s ** (d_head / (d_head - 2))

In [18]:
@torch.no_grad()
def eval_ppl(model: torch.nn.Module, ids: torch.Tensor, ctx_len: int, stride: int = 1) -> float:
    if ids.ndim != 2 or ids.size(0) == 0 or ids.size(1) < 2:
        raise ValueError("pass a nonempty [batch, length] tensor with length >= 2")
    if ctx_len < 2 or not 1 <= stride < ctx_len:
        raise ValueError("use ctx_len >= 2 and 1 <= stride < ctx_len")
    training = model.training
    model.eval()
    nll_sum, n_tok, prev_end = 0.0, 0, 1
    try:
        for begin in range(0, ids.size(1) - 1, stride):
            end = min(begin + ctx_len, ids.size(1))
            output = model(ids[:, begin:end])
            logits = output if isinstance(output, torch.Tensor) else output.logits
            first = max(prev_end, begin + 1)
            targets = ids[:, first:end]
            scores = logits[:, first - begin - 1:end - begin - 1]
            nll_sum += F.cross_entropy(
                scores.reshape(-1, scores.size(-1)), targets.reshape(-1), reduction="sum"
            ).item()
            n_tok += targets.numel()
            prev_end = end
            if end == ids.size(1):
                break
    finally:
        model.train(training)
    return math.exp(nll_sum / n_tok)

In [19]:
import math
from types import SimpleNamespace


import torch


set_seed(0)
L_train, L_eval, D, s = 256, 1024, 64, 4
freq = 10000.0 ** (-torch.arange(0, D, 2).float() / D)
angle = (L_eval - 1) * freq[-1]
show('raw low-frequency angle', float(angle))
show('PI low-frequency angle', float(angle / s))
base = ntk_base(10000.0, D, s)
scaled = base ** (-torch.arange(0, D, 2).float() / D)
show('NTK base', base)
show('NTK high/low freq ratio', (scaled / freq)[[0, -1]].tolist())


class Bigram(torch.nn.Module):
    def __init__(self, vocab: int):
        super().__init__()
        self.register_buffer('table', torch.eye(vocab) * 2)

    def forward(self, ids: torch.Tensor):
        return SimpleNamespace(logits=self.table[ids])


ids = torch.randint(0, 16, (2, 257))
model = Bigram(16)
for ctx in [16, 64, 256]:
    show(f'bigram ppl ctx={ctx}', eval_ppl(model, ids, ctx, stride=min(8, ctx-1)))

raw low-frequency angle      0.13641923666000366
PI low-frequency angle       0.034104809165000916
NTK base                     41829.36592889948
NTK high/low freq ratio      [1.0, 0.25]
bigram ppl ctx=16            19.376121501478647
bigram ppl ctx=64            19.376120924025233
bigram ppl ctx=256           19.37612568801639


滑窗评估让每行 target 索引 1..T-1 恰好计分一次；普通 causal LM 的 logits[t] 预测 token[t+1]，这不同于本项目 TinyLM 内部显式 BOS shift 的约定。固定 bigram 在不同 context 得到相同 PPL 只验证记账，不是长上下文能力测量。YaRN、decoupled RoPE、多维 PE 在文章中介绍，但本核心未实现，不能据此声称覆盖全部位置技术。

## 7. NoPE：mask 可见数量是否就是位置？

因果 mask 使第 t 行可见 t+1 个 key，因此结构中可能携带顺序线索。

In [20]:
L = 8
visible = causal_mask(L).isfinite().sum(dim=-1)
show("visible keys by position", visible.tolist())

visible keys by position     [1, 2, 3, 4, 5, 6, 7, 8]


In [21]:
q = k = torch.zeros(1, 8, 4)
v = torch.ones(1, 8, 4)
y = attention(q, k, v, causal_mask(8))
show('identical values stay equal', torch.allclose(y, v))

identical values stay equal  True


反例中虽然可见数量不同，完全相同的 value 平均后仍相同。因此 mask visibility 可以携带顺序，但它本身不是“网络一定恢复绝对位置”的证明。